# NB06 — Validation, Regularisation, Threshold Tuning and Business Interpretation

**Role:** Core  
**Manual section:** 2.3.0 — Validation and model assessment; bridges to 2.3.1.b  
**Prerequisites:** NB05b

---

## Purpose of this notebook

This notebook closes the core modelling workflow. You will learn cross-validation, class imbalance handling, precision-recall trade-offs with financial cost framing, threshold tuning, regularisation, GridSearchCV, feature importance and executive recommendation writing. Every concept is translated into business language.

**Dataset:** `dataset_C_attrition_model_ready.csv`

---

## Section 0 — Why validation matters

In Notebooks 05 and 05b we trained several models and compared their metrics on a
single train/test split. The results looked promising:

| Model | AUC-ROC | Recall |
|-------|---------|--------|
| Logistic Regression | 0.871 | 0.447 |
| Decision Tree (depth=4) | 0.860 | 0.511 |
| Random Forest | 0.910 | 0.523 |
| XGBoost | 0.911 | 0.624 |

But a single evaluation is **not enough** to trust a model. Why?

1. **Stability** — did we just get lucky with the random split?
2. **Trust** — would the model behave similarly on tomorrow's customers?
3. **Governance** — regulators expect proof of robustness, not a single number.
4. **Production risk** — a model that works on one sample but fails on another is
   worse than no model at all.

This notebook adds **validation discipline**, **threshold awareness** and
**business interpretation** — the layer that turns a score into a decision.

---

## Section 1 — Rebuild the best candidate models

We rebuild the modelling table and re-train the three key models from the earlier
notebooks so everything is self-contained and reproducible.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
import warnings
from pathlib import Path

# Reproducibility
SEED = 42
np.random.seed(SEED)

# Resolve data directory — works from the project root or notebooks/
_candidates = [Path("data/processed"), Path("../data/processed")]
DATA_DIR = next((p for p in _candidates if p.is_dir()), None)
if DATA_DIR is None:
    raise FileNotFoundError(
        "Cannot locate data/processed/. "
        "Launch the notebook from the project root or the notebooks/ folder."
    )

# ── Load dataset ──────────────────────────────────────────────────────
df = pd.read_csv(DATA_DIR / "dataset_C_bank_attrition_core.csv")

# ── One-hot encoding (same as NB04) ──────────────────────────────────
df_encoded = pd.get_dummies(df, columns=["geography", "gender"], drop_first=True)

target_col = "attrition_flag"
id_col     = "customer_id"

y = df_encoded[target_col]
X = df_encoded.drop(columns=[target_col, id_col])

print(f"Dataset: {df.shape[0]:,} rows × {df.shape[1]} columns")
print(f"Features: {X.shape[1]}  |  Target: {target_col}")
print(f"Churn rate: {y.mean():.3f}")

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score,
    f1_score, roc_auc_score, confusion_matrix,
    classification_report, ConfusionMatrixDisplay,
)

# ── Train / test split (identical to NB04) ───────────────────────────
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=SEED, stratify=y,
)

# ── Scaled version for logistic regression ───────────────────────────
scaler = StandardScaler()
X_train_sc = pd.DataFrame(
    scaler.fit_transform(X_train), columns=X_train.columns, index=X_train.index
)
X_test_sc = pd.DataFrame(
    scaler.transform(X_test), columns=X_test.columns, index=X_test.index
)

print(f"Training: {X_train.shape[0]:,} rows × {X_train.shape[1]} features")
print(f"Test:     {X_test.shape[0]:,} rows × {X_test.shape[1]} features")
print(f"Churn rate — train: {y_train.mean():.3f}, test: {y_test.mean():.3f}")

In [ ]:
# ── 1. Logistic Regression baseline ──────────────────────────────────
logreg = LogisticRegression(max_iter=1000, random_state=SEED)
logreg.fit(X_train_sc, y_train)

# ── 2. Random Forest ─────────────────────────────────────────────────
rf = RandomForestClassifier(
    n_estimators=200, max_depth=8, random_state=SEED, n_jobs=-1
)
rf.fit(X_train, y_train)

# ── 3. XGBoost ───────────────────────────────────────────────────────
xgb = XGBClassifier(
    n_estimators=200, max_depth=5, learning_rate=0.1,
    subsample=0.8, colsample_bytree=0.8,
    random_state=SEED, eval_metric="logloss",
)
xgb.fit(X_train, y_train)

print("Three candidate models trained: LogReg, Random Forest, XGBoost.")

> We intentionally rebuild all three models here so that this notebook can be
> run independently without relying on earlier notebook state.

---

## Section 2 — Train vs test behaviour and overfitting signs

A model that performs brilliantly on training data but poorly on test data has
**memorised** rather than **learned**. This is called *overfitting*.

The simplest diagnostic: compare common metrics on both sets and look at the
**gap**. A small gap is healthy; a large gap is a warning sign.

In [ ]:
def eval_set(model, X_tr, X_te, y_tr, y_te, name, scale=False):
    """Compute train and test metrics for a given model."""
    Xtr = X_tr if not scale else X_train_sc
    Xte = X_te if not scale else X_test_sc
    metrics = {}
    for label, Xs, ys in [("Train", Xtr, y_tr), ("Test", Xte, y_te)]:
        yp = model.predict(Xs)
        ypr = model.predict_proba(Xs)[:, 1]
        metrics[label] = {
            "Accuracy": accuracy_score(ys, yp),
            "Precision": precision_score(ys, yp),
            "Recall": recall_score(ys, yp),
            "F1": f1_score(ys, yp),
            "AUC-ROC": roc_auc_score(ys, ypr),
        }
    row = {"Model": name}
    for m in ["Accuracy", "Precision", "Recall", "F1", "AUC-ROC"]:
        row[f"{m} (Train)"] = metrics["Train"][m]
        row[f"{m} (Test)"]  = metrics["Test"][m]
        row[f"{m} Gap"]     = metrics["Train"][m] - metrics["Test"][m]
    return row

rows = [
    eval_set(logreg, X_train, X_test, y_train, y_test, "Logistic Regression", scale=True),
    eval_set(rf,     X_train, X_test, y_train, y_test, "Random Forest"),
    eval_set(xgb,    X_train, X_test, y_train, y_test, "XGBoost"),
]
gap_df = pd.DataFrame(rows).set_index("Model")

# Show the key AUC and Recall gaps
summary_cols = ["AUC-ROC (Train)", "AUC-ROC (Test)", "AUC-ROC Gap",
                "Recall (Train)", "Recall (Test)", "Recall Gap"]
print("Train vs Test Comparison")
print("=" * 60)
print(gap_df[summary_cols].round(3).to_string())

### What to look for

| Signal | Interpretation |
|--------|---------------|
| **AUC Gap < 0.02** | Healthy — the model generalises well |
| **AUC Gap 0.02 – 0.05** | Moderate — keep an eye on it |
| **AUC Gap > 0.05** | Warning — the model may be overfitting |
| **Recall Gap > 0.10** | The model catches attrition cases in training but misses them in test — unreliable |

> **Reflection:** logistic regression typically has the smallest gap (low
> complexity = low overfitting risk). XGBoost and Random Forest are more
> powerful but must be checked for overfitting — *power and discipline go
> hand in hand*.

---

## Section 3 — Class imbalance awareness

Our target has approximately **80 % stayed / 20 % attrited**. This ratio matters
because a lazy model that predicts *"stayed"* for every customer would still be
80 % accurate — yet it would be completely useless for identifying attrition (churn) risk.

This is why we need metrics beyond accuracy.

In [ ]:
# Class distribution
counts = y_test.value_counts()
print("Test set class distribution")
print("-" * 35)
for val, cnt in counts.items():
    label = "Churned" if val == 1 else "Stayed"
    print(f"  {label} ({val}): {cnt:>5}  ({cnt / len(y_test):.1%})")
print()

# Illustrate the "always predict stayed" baseline
naive_preds = pd.Series(0, index=y_test.index)
print(f"Naive 'always stayed' baseline:")
print(f"  Accuracy = {accuracy_score(y_test, naive_preds):.3f}")
print(f"  Recall   = {recall_score(y_test, naive_preds):.3f}  ← catches ZERO churners")
print(f"  F1       = {f1_score(y_test, naive_preds):.3f}")
print()
print("→ Accuracy alone can be dangerously misleading.")

### Metric refresher for imbalanced targets

| Metric | Meaning in our context | Why it matters |
|--------|----------------------|----------------|
| **Precision** | Of customers we flag as attrition cases, how many actually attrite? | Controls **false positives** — wasted retention spend |
| **Recall** | Of all real attrition cases, how many do we catch? | Controls **false negatives** — lost customers we failed to help |
| **F1** | Harmonic mean of precision and recall | Balances both concerns |
| **AUC-ROC** | Overall ranking ability across all thresholds | Threshold-independent quality measure |

> In a **customer retention** scenario, missing a real attrition case (low recall) often
> costs more than wrongly contacting a happy customer (low precision). But the
> right balance depends on the business context and the cost of each action.

> **From confusion matrix to financial cost**  
> In finance, a confusion matrix is not just a technical summary — it is a cost map.  
> **False positives** may mean unnecessary retention calls or review costs.  
> **False negatives** may mean missed attrition cases, undetected fraud, or lost revenue.  
>  
> **Precision** tells us how much of our flagged population is truly relevant: it behaves like an **efficiency / risk-control** measure.  
> **Recall** tells us how much of the real positive class we actually capture: it behaves like a **business capture** measure.  
> **F1 score** gives a single compromise when we need both to be reasonable at the same time.

---

## Section 4 — Threshold tuning: from default to business-aligned

### The hidden assumption

Every classification model outputs a **probability** (e.g., 0.37 or 0.72). To
make a decision — *"flag this customer as attrition risk or not"* — we compare that
probability to a **threshold**.

By default, scikit-learn and XGBoost use **threshold = 0.5**:

- probability ≥ 0.5 → predict *attrited*
- probability < 0.5 → predict *stayed*

But **0.5 is not a law of nature**. It is just a convenient default. In practice,
the right threshold depends on the **relative cost** of two types of mistakes:

| Mistake | Name | Business impact |
|---------|------|-----------------|
| Predict "stayed" but customer attrits | **False Negative** | Lost revenue — we missed a customer we could have retained |
| Predict "attrited" but customer stays | **False Positive** | Wasted retention budget — we spent resources on a loyal customer |

> **Key insight:** lowering the threshold catches more attrition cases (↑ recall) but
> also sends more false alarms (↓ precision). Raising the threshold does the
> opposite. There is no free lunch — only a **business trade-off**.

In [ ]:
# ── Threshold sweep on XGBoost (our strongest model) ─────────────────
y_prob_xgb = xgb.predict_proba(X_test)[:, 1]

thresholds = np.arange(0.10, 0.91, 0.05)
threshold_rows = []
for t in thresholds:
    y_pred_t = (y_prob_xgb >= t).astype(int)
    threshold_rows.append({
        "Threshold": round(t, 2),
        "Precision": precision_score(y_test, y_pred_t, zero_division=0),
        "Recall":    recall_score(y_test, y_pred_t, zero_division=0),
        "F1":        f1_score(y_test, y_pred_t, zero_division=0),
        "Flagged":   y_pred_t.sum(),
    })

thresh_df = pd.DataFrame(threshold_rows).round(3)
print(thresh_df.to_string(index=False))

In [ ]:
# ── Precision–Recall trade-off chart ──────────────────────────────────
fig, ax1 = plt.subplots(figsize=(9, 5))

ax1.plot(thresh_df["Threshold"], thresh_df["Precision"],
         marker="o", label="Precision", color="#5b9bd5")
ax1.plot(thresh_df["Threshold"], thresh_df["Recall"],
         marker="s", label="Recall", color="#ed7d31")
ax1.plot(thresh_df["Threshold"], thresh_df["F1"],
         marker="^", label="F1", linestyle="--", color="#70ad47")

ax1.axvline(0.50, color="grey", linestyle=":", label="Default threshold (0.50)")
ax1.axvline(0.35, color="red",  linestyle=":", alpha=0.7, label="Business threshold (0.35)")
ax1.set_xlabel("Threshold")
ax1.set_ylabel("Score")
ax1.set_title("Precision, Recall and F1 vs Classification Threshold (XGBoost)")
ax1.legend(loc="center left")
ax1.set_ylim(0, 1.05)
ax1.grid(axis="y", alpha=0.3)
plt.tight_layout()
plt.show()

### Reading the chart

- At **threshold = 0.50** (default): precision is relatively high but recall is
  lower — we miss many real attrition cases.
- At **threshold = 0.35** (illustrative business choice): recall jumps noticeably
  — we catch more attrition cases, but precision drops — more false alarms.
  *Note: 0.35 is not a universal rule. The right threshold depends entirely on
  the cost structure of each specific business context.*
- The **F1 curve** peaks somewhere between 0.30 and 0.45, suggesting a good
  balance point.

> **Business question:** *"Is it more expensive to miss an attrition case or to contact a
> loyal customer unnecessarily?"*
>
> If retention outreach costs €50 per customer and losing an attrition case costs €500 in
> lifetime value, then false negatives are **10× more expensive** than false
> positives. In that case, a lower threshold (higher recall) makes business
> sense.

In [ ]:
# ── Compare two concrete thresholds ──────────────────────────────────
def threshold_report(y_true, y_prob, t, label):
    y_pred = (y_prob >= t).astype(int)
    cm = confusion_matrix(y_true, y_pred)
    tn, fp, fn, tp = cm.ravel()
    return {
        "Threshold": label,
        "True Pos (caught churners)":  tp,
        "False Neg (missed churners)": fn,
        "False Pos (false alarms)":    fp,
        "True Neg":                    tn,
        "Precision": precision_score(y_true, y_pred, zero_division=0),
        "Recall":    recall_score(y_true, y_pred, zero_division=0),
        "F1":        f1_score(y_true, y_pred, zero_division=0),
    }

t_default  = threshold_report(y_test, y_prob_xgb, 0.50, "0.50 (default)")
t_business = threshold_report(y_test, y_prob_xgb, 0.35, "0.35 (business)")

compare_df = pd.DataFrame([t_default, t_business]).set_index("Threshold")
print(compare_df.to_string())

### The cost perspective

Suppose:
- Retention action cost = **€50** per flagged customer
- Cost of losing an attrition case = **€500** (lost revenue over 2 years)

| Threshold | False Alarms (FP × €50) | Missed Attrition Cases (FN × €500) | Total estimated cost |
|-----------|------------------------|----------------------------|---------------------|

*(Fill in the numbers from the output above as a practice exercise.)*

> **Key takeaway:** threshold tuning is not a statistical nicety — it is a
> **business decision** that directly affects how much the model costs or saves.
> The "best" threshold is the one that minimises total business cost, not the one
> that maximises a single metric.

---

## Section 5 — Confusion matrix at the chosen threshold

A confusion matrix shows the four possible outcomes for every prediction. It is
the most concrete way to see what a model actually does.

In [ ]:
# ── Confusion matrices side by side ──────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

for ax, t, title in [
    (axes[0], 0.50, "Threshold = 0.50 (default)"),
    (axes[1], 0.35, "Threshold = 0.35 (business)"),
]:
    y_pred_t = (y_prob_xgb >= t).astype(int)
    ConfusionMatrixDisplay.from_predictions(
        y_test, y_pred_t,
        display_labels=["Stayed", "Churned"],
        cmap="Blues", ax=ax,
    )
    ax.set_title(title)

plt.tight_layout()
plt.show()

### How to read the confusion matrix

|  | Predicted Stayed | Predicted Attrited |
|--|-----------------|------------------|
| **Actually Stayed** | True Negative (correct) | False Positive (false alarm) |
| **Actually Attrited** | False Negative (missed!) | True Positive (caught) |

- **Top-right** = false alarms → retention budget wasted
- **Bottom-left** = missed attrition cases → revenue lost
- Moving from threshold 0.50 to 0.35 shifts cases from bottom-left to top-right:
  we catch more attrition cases but accept more false alarms.

> The confusion matrix makes the precision–recall trade-off **visible and
> concrete**.

---

## Section 6 — Regularisation intuition

### What is regularisation?

Regularisation is a technique that **penalises model complexity** to prevent
overfitting. Think of it as adding a "simplicity preference" to the model:

> *"If two models perform equally well on training data, prefer the simpler one."*

In everyday language:
- A model without regularisation tries to fit **every detail** in the training
  data, including noise and one-off patterns.
- A regularised model accepts a slightly worse training fit in exchange for
  **better generalisation** to new data.

### How it works in practice

| Model family | Regularisation mechanism | Intuitive effect |
|-------------|------------------------|-----------------|
| **Logistic Regression** | Penalty on large coefficients (L1 or L2) | Prevents any single feature from dominating the prediction |
| **XGBoost / boosted trees** | `max_depth`, `learning_rate`, `reg_alpha`, `reg_lambda` | Controls how deep and aggressive each tree can be |
| **Random Forest** | `max_depth`, `min_samples_leaf` | Prevents individual trees from growing to extreme depth |

> **Analogy:** regularisation is like a study plan that says *"don't memorise the
> textbook word by word — understand the key concepts so you can answer unseen
> exam questions."*

In [ ]:
# ── Regularisation demo: logistic regression with different C values ──
from sklearn.pipeline import make_pipeline

reg_results = []
for C_val in [0.001, 0.01, 0.1, 1.0, 10.0, 100.0]:
    model = make_pipeline(
        StandardScaler(),
        LogisticRegression(C=C_val, max_iter=1000, random_state=SEED)
    )
    model.fit(X_train, y_train)
    y_tr_pred = model.predict(X_train)
    y_te_pred = model.predict(X_test)
    y_te_prob = model.predict_proba(X_test)[:, 1]
    reg_results.append({
        "C": C_val,
        "Train Acc": accuracy_score(y_train, y_tr_pred),
        "Test Acc":  accuracy_score(y_test, y_te_pred),
        "Test AUC":  roc_auc_score(y_test, y_te_prob),
        "Test Recall": recall_score(y_test, y_te_pred),
    })

reg_df = pd.DataFrame(reg_results)
print("Logistic Regression — effect of regularisation strength (C)")
print("(Lower C = stronger regularisation)")
print()
print(reg_df.round(4).to_string(index=False))

### Reading the results

- **C = 0.001** (very strong regularisation): the model is overly simple — it
  under-fits and test performance suffers.
- **C = 1.0** (scikit-learn default): a reasonable balance.
- **C = 100** (very weak regularisation): the model has almost no penalty — it
  fits training data closely and may overfit.

> The ideal C sits in a *"Goldilocks zone"* — not too simple, not too complex.
> Finding it is part of **hyperparameter tuning** (next section).

---

## Section 7 — Cross-validation for robustness

A single train/test split gives us one estimate of performance. But how **stable**
is that estimate? Cross-validation answers this by repeating the evaluation on
multiple different splits.

### How 5-fold cross-validation works

1. Split the training data into 5 equal parts ("folds").
2. Train on 4 folds, evaluate on the 1 held-out fold.
3. Repeat 5 times, each time holding out a different fold.
4. Report the **mean** and **standard deviation** of each metric.

> A model with high mean AUC but large std is **unstable** — it might perform
> well on some splits and poorly on others. We want **high mean + low std**.

In [ ]:
from sklearn.model_selection import cross_val_score

# Cross-validate each model on AUC-ROC
cv_results = {}

# Logistic Regression (needs scaling — use pipeline)
lr_pipe = make_pipeline(
    StandardScaler(),
    LogisticRegression(max_iter=1000, random_state=SEED)
)
scores_lr = cross_val_score(lr_pipe, X_train, y_train, cv=5, scoring="roc_auc")
cv_results["Logistic Regression"] = scores_lr

# Random Forest
scores_rf = cross_val_score(rf, X_train, y_train, cv=5, scoring="roc_auc")
cv_results["Random Forest"] = scores_rf

# XGBoost
scores_xgb = cross_val_score(xgb, X_train, y_train, cv=5, scoring="roc_auc")
cv_results["XGBoost"] = scores_xgb

print("5-Fold Cross-Validation — AUC-ROC")
print("=" * 55)
for name, scores in cv_results.items():
    print(f"  {name:25s}  mean={scores.mean():.3f}  std={scores.std():.3f}  "
          f"folds={[round(float(s), 3) for s in scores]}")

### Interpreting the results

| What to check | Good sign | Warning sign |
|---------------|-----------|-------------|
| **Mean AUC** | > 0.85 | < 0.75 |
| **Std of AUC** | < 0.02 | > 0.04 |
| **Consistency** | All folds within ±0.02 of mean | One fold much lower than the rest |

> If XGBoost has the highest mean AUC **and** a low standard deviation, we have
> stronger evidence that its performance is real — not just an artefact of one
> lucky split.

---

## Section 8 — Lightweight hyperparameter tuning

Hyperparameters are settings we choose **before** training — like `max_depth`,
`learning_rate` or `n_estimators`. The model cannot learn these from data; we
must choose them ourselves.

In a classroom setting, we keep this simple:
- Try a small grid of sensible values.
- Use cross-validation to compare them.
- Pick the best combination.

We focus on XGBoost since it was our strongest candidate.

In [ ]:
from sklearn.model_selection import GridSearchCV

# Small, classroom-friendly parameter grid
param_grid = {
    "max_depth":     [3, 5, 7],
    "learning_rate": [0.05, 0.10],
    "n_estimators":  [100, 200],
}

grid_search = GridSearchCV(
    XGBClassifier(
        subsample=0.8, colsample_bytree=0.8,
        random_state=SEED, eval_metric="logloss",
    ),
    param_grid,
    cv=5,
    scoring="roc_auc",
    n_jobs=-1,
    refit=True,
)
grid_search.fit(X_train, y_train)

print(f"Best parameters: {grid_search.best_params_}")
print(f"Best CV AUC-ROC: {grid_search.best_score_:.4f}")

In [ ]:
# Show all grid results sorted by AUC
grid_df = pd.DataFrame(grid_search.cv_results_)
grid_df = grid_df.sort_values("rank_test_score")
cols = ["param_max_depth", "param_learning_rate", "param_n_estimators",
        "mean_test_score", "std_test_score", "rank_test_score"]
print(grid_df[cols].head(12).to_string(index=False))

### Observations

- The differences between configurations are often **small** — this is typical
  in practice.
- A simpler configuration that performs nearly as well as the best may be
  preferred for **interpretability** and **stability**.
- Hyperparameter tuning is useful but should not be over-engineered: a 0.002
  AUC improvement is rarely worth significant added complexity.

> **Rule of thumb:** if a simpler setting gives AUC within 0.005 of the best,
> prefer the simpler one.

---

## Section 9 — Evaluate the tuned model with business threshold

Let us now evaluate the grid-search winner on the **test set** at both the
default and business thresholds.

In [ ]:
# ── Best model from grid search ───────────────────────────────────────
best_xgb = grid_search.best_estimator_
y_prob_best = best_xgb.predict_proba(X_test)[:, 1]

# Metrics at two thresholds
for t, label in [(0.50, "Default (0.50)"), (0.35, "Business (0.35)")]:
    y_pred_t = (y_prob_best >= t).astype(int)
    print(f"\nThreshold: {label}")
    print(f"  Accuracy:  {accuracy_score(y_test, y_pred_t):.3f}")
    print(f"  Precision: {precision_score(y_test, y_pred_t, zero_division=0):.3f}")
    print(f"  Recall:    {recall_score(y_test, y_pred_t, zero_division=0):.3f}")
    print(f"  F1:        {f1_score(y_test, y_pred_t, zero_division=0):.3f}")
    print(f"  AUC-ROC:   {roc_auc_score(y_test, y_prob_best):.3f}")

# Confusion matrix for business threshold
print("\nConfusion Matrix at threshold 0.35:")
y_pred_biz = (y_prob_best >= 0.35).astype(int)
cm = confusion_matrix(y_test, y_pred_biz)
print(pd.DataFrame(cm,
    index=["Actually Stayed", "Actually Churned"],
    columns=["Predicted Stayed", "Predicted Churned"],
))

> Compare these numbers with the original XGBoost (NB05b). Tuning and
> threshold adjustment together can improve **practical usefulness** even if
> the AUC barely changes — because we are now optimising for the **right
> business metric**, not just accuracy.

> **A note on AUC and calibration:** a strong AUC-ROC means the model
> *ranks* attrition cases above non-attrition cases effectively, but it does **not**
> guarantee that the predicted probabilities are well-calibrated (i.e.,
> that a predicted 0.35 truly corresponds to a 35 % attrition chance).
> In production, probability calibration (e.g., Platt scaling or isotonic
> regression) may be needed before using raw scores for threshold-based
> decisions.

---

## Section 10 — Interpretation beyond raw score

Feature importance tells us which variables the model relies on most, but
**importance is not causality**.

- A feature that ranks high may be a proxy for something else.
- Importance varies by method (gain, permutation, SHAP).
- Business context must validate what the model suggests.

In [ ]:
# ── Feature importance from the tuned XGBoost ────────────────────────
fi = pd.DataFrame({
    "feature": X_train.columns,
    "importance": best_xgb.feature_importances_,
}).sort_values("importance", ascending=True)

fig, ax = plt.subplots(figsize=(8, 5))
ax.barh(fi["feature"], fi["importance"], color="#70ad47", edgecolor="white")
ax.set_xlabel("Importance (gain)")
ax.set_title("XGBoost (Tuned) — Feature Importance")
plt.tight_layout()
plt.show()

### Interpretation checklist

Before trusting feature importance, ask:

1. **Does the ranking make business sense?** If `age` and `is_active_member`
   rank high, that aligns with banking intuition about attrition drivers.
2. **Is the ranking stable?** Run the model with a different random seed or
   slightly different parameters. If the ranking changes dramatically, it is
   not robust.
3. **Does the feature cause the outcome?** Correlation ≠ causation. A feature
   may be a proxy or a confound.

> **For stakeholders:** present the top 3–5 features with plain-language
> explanations, always noting that *"these are the features the model found most
> useful, not necessarily the root causes of attrition."*

---

## Section 11 — Bivariate analysis: understanding features one by one

Before applying any interpretability constraint to a model, the analyst needs to see
**how each feature relates to the target**. A bivariate plot answers the question:
*"When this feature increases, does the attrition rate go up, go down, or behave
non-monotonically?"*

This matters in finance because:

- Regulators and auditors expect the **direction of effect** to be explainable.
- Business logic sometimes imposes hard expectations (e.g., longer tenure should
  reduce attrition risk, not increase it).
- Bivariates reveal whether a monotonicity constraint (Section 12) is justified.

The technique is simple:
1. **For continuous features:** bin the values into quantiles and compute the
   attrition rate within each bin.
2. **For categorical features:** compute the attrition rate per category.
3. **Plot:** frequency (bar) + attrition rate (line) on a dual-axis chart.

In [ ]:
def bivariate_plot(data, feature, target, n_bins=7, ax=None):
    """
    Bivariate analysis: frequency (bar) + mean target rate (line).
    
    For continuous features, bins into quantiles.
    For categorical/binary features, uses the values directly.
    """
    if ax is None:
        fig, ax = plt.subplots(figsize=(6, 3.5))
    
    temp = data[[feature, target]].copy()
    nunique = temp[feature].nunique()
    
    # Decide whether to bin: continuous if more than 10 unique values
    if nunique > 10:
        temp["bin"] = pd.qcut(temp[feature], q=n_bins, duplicates="drop")
        group_col = "bin"
    else:
        group_col = feature
    
    grouped = temp.groupby(group_col, observed=True).agg(
        count=(target, "size"),
        rate=(target, "mean"),
    )
    
    # Bar chart: frequency
    colour_bar = "#2E5C8A"
    colour_line = "#ED7D31"
    ax.bar(range(len(grouped)), grouped["count"], color=colour_bar, alpha=0.6, label="Count")
    ax.set_ylabel("Frequency", color=colour_bar)
    ax.set_xticks(range(len(grouped)))
    labels = [str(v)[:18] for v in grouped.index]
    ax.set_xticklabels(labels, rotation=45, ha="right", fontsize=8)
    ax.set_title(feature, fontsize=11, fontweight="bold")
    
    # Line chart: attrition rate on twin axis
    ax2 = ax.twinx()
    ax2.plot(range(len(grouped)), grouped["rate"], color=colour_line,
             marker="o", linewidth=2, label="Attrition rate")
    ax2.set_ylabel("Attrition rate", color=colour_line)
    ax2.set_ylim(0, min(grouped["rate"].max() * 1.5, 1.0))
    
    return grouped

In [ ]:
# ── Apply bivariate analysis to the top 8 features ───────────────────
# Use the feature importance ranking from Section 10
top_features = fi.sort_values("importance", ascending=False).head(8)["feature"].tolist()

# Prepare a merged DataFrame (training data with target for analysis)
biv_df = X_train.copy()
biv_df[target_col] = y_train.values

fig, axes = plt.subplots(2, 4, figsize=(18, 9))
axes = axes.flatten()

for i, feat in enumerate(top_features):
    bivariate_plot(biv_df, feat, target_col, ax=axes[i])

plt.suptitle("Bivariate Analysis — Top 8 Features vs Attrition Rate", fontsize=13, y=1.02)
plt.tight_layout()
plt.show()

### Interpreting the bivariates and deriving monotonicity constraints

Examine each plot above. For each feature, ask:
- **Is the attrition rate clearly increasing as the feature increases?** → constraint = **+1** (monotonically increasing)
- **Is the attrition rate clearly decreasing?** → constraint = **−1** (monotonically decreasing)
- **Is the relationship non-monotonic or unclear?** → constraint = **0** (unconstrained)

The table below records the observed direction. Your exact values may vary slightly
depending on the random seed, but the general patterns should be stable.

| Feature | Observed direction | Constraint |
|---------|-------------------|------------|
| age | ↑ older → more attrition | +1 |
| is_active_member | ↓ active members attrite less | −1 |
| balance | non-monotonic (low and high balance attrite more) | 0 |
| num_of_products | non-monotonic | 0 |
| complaints_last_6m | ↑ more complaints → more attrition | +1 |
| digital_usage_proxy | ↓ more digital engagement → less attrition | −1 |
| credit_score | ↓ higher score → less attrition (weak) | −1 |
| estimated_salary | weak / unclear | 0 |

> **Key insight:** bivariate analysis is done **before** modelling constraints
> are applied. It provides the empirical basis for monotonicity decisions —
> the analyst does not guess the direction, they observe it in data.

---

## Section 12 — Monotonicity constraints in XGBoost

### Why monotonicity matters in finance

In regulated finance — credit scoring, insurance pricing, anti-money laundering —
models often must respect **domain knowledge**. For example:

- A customer with *more transactions* should not be scored as *higher* attrition
  risk if the data clearly shows the opposite.
- A borrower with *higher income* should not receive a *worse* credit score
  unless there is a very good reason.

When a model violates these expectations, it becomes hard to explain to auditors
and dangerous to deploy. **Monotonicity constraints** force XGBoost to respect
the observed direction of each feature's relationship with the target.

### How `monotone_constraints` works

XGBoost accepts a tuple with one value per feature:

| Value | Meaning |
|-------|---------|
| **+1** | The model's prediction must **increase** (or stay flat) as this feature increases |
| **−1** | The model's prediction must **decrease** (or stay flat) as this feature increases |
| **0** | No constraint — the model can learn any shape |

The constraint is enforced during tree construction — splits that would violate
the direction are simply not allowed. This may reduce predictive power slightly,
but the gain in **coherence and explainability** is often worth it.

In [ ]:
# ── Build the monotonicity constraint vector ─────────────────────────
# Map: feature name → constraint direction (from bivariate analysis)
# Only clearly monotonic features are constrained; ambiguous ones default to 0.

constraint_map = {
    "age":                  +1,   # older → more attrition
    "is_active_member":     -1,   # active → less attrition
    "complaints_last_6m":   +1,   # more complaints → more attrition
    "digital_usage_proxy":  -1,   # more digital engagement → less attrition
    "credit_score":         -1,   # higher score → less attrition
    # balance, num_of_products, estimated_salary, tenure: unconstrained (0)
}

# Build the tuple in the same order as the feature columns
monotone_constraints = tuple(
    constraint_map.get(col, 0) for col in X_train.columns
)

print(f"Features: {len(monotone_constraints)}")
print(f"Constrained: {sum(1 for c in monotone_constraints if c != 0)}")
print(f"\nConstraint vector:")
for col, c in zip(X_train.columns, monotone_constraints):
    if c != 0:
        direction = "↑ increasing" if c == +1 else "↓ decreasing"
        print(f"  {col:25s} → {c:+d}  ({direction})")
    else:
        print(f"  {col:25s} →  0  (unconstrained)")

In [ ]:
# ── Train XGBoost with monotonicity constraints ─────────────────────
# Use the same hyperparameters as the best unconstrained model
xgb_constrained = XGBClassifier(
    **grid_search.best_params_,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=SEED,
    eval_metric="logloss",
    monotone_constraints=monotone_constraints,
)
xgb_constrained.fit(X_train, y_train)

y_prob_constrained = xgb_constrained.predict_proba(X_test)[:, 1]

# ── Compare constrained vs unconstrained at threshold 0.35 ──────────
print("Comparison: Unconstrained vs Monotonicity-Constrained XGBoost")
print("=" * 65)

for name, probs in [("Unconstrained", y_prob_best), ("Constrained", y_prob_constrained)]:
    y_pred_t = (probs >= 0.35).astype(int)
    print(f"\n  {name}:")
    print(f"    AUC-ROC:   {roc_auc_score(y_test, probs):.4f}")
    print(f"    Precision: {precision_score(y_test, y_pred_t, zero_division=0):.3f}")
    print(f"    Recall:    {recall_score(y_test, y_pred_t, zero_division=0):.3f}")
    print(f"    F1:        {f1_score(y_test, y_pred_t, zero_division=0):.3f}")

### Interpretation

The constrained model typically shows a **small** AUC decrease (often < 0.01) compared to
the unconstrained version. This is the cost of interpretability: the model can no longer
exploit non-monotonic patterns that might be noise or artefacts.

The benefit is substantial:

- Every feature now behaves in the **expected direction** — easier to explain to
  auditors, regulators and business stakeholders.
- The model is **more robust** — monotonicity acts as an implicit regulariser,
  reducing overfitting to spurious patterns.
- In credit scoring and insurance, monotonicity constraints are often a
  **regulatory requirement**, not just a nice-to-have.

> **Trade-off:** predictive power decreases slightly, but coherence and
> explainability increase significantly. In most regulated finance settings,
> this is the right trade-off.

---

## Section 13 — SHAP values: explaining individual predictions

### From global importance to individual explanation

Section 10 showed **gain-based feature importance** — a global summary of which
features the model uses most. But gain tells you nothing about:

- **How** a feature affects a specific prediction (positively or negatively).
- **Why** a particular customer was flagged as high risk.
- Whether two models use the same features in the **same way**.

**SHAP (SHapley Additive exPlanations)** solves this. It comes from cooperative
game theory: for each prediction, SHAP computes how much each feature
*contributed* to pushing the prediction above or below the average.

### Three levels of SHAP explanation

| Level | Question answered | Plot type |
|-------|------------------|-----------|
| **Global** | "What drives attrition overall?" | Summary / beeswarm plot |
| **Feature-level** | "How does age affect attrition across all customers?" | Dependence plot |
| **Individual** | "Why was *this* customer flagged?" | Waterfall plot |

For tree-based models like XGBoost, SHAP provides an **exact** and efficient
computation via `TreeExplainer`.

In [ ]:
import shap

# ── SHAP for the unconstrained (tuned) model ────────────────────────
explainer_unc = shap.TreeExplainer(best_xgb)
shap_values_unc = explainer_unc(X_test)

# ── Global summary: beeswarm plot ───────────────────────────────────
print("SHAP Summary — Unconstrained XGBoost")
shap.summary_plot(shap_values_unc, X_test, show=False)
plt.title("SHAP Summary — Unconstrained XGBoost", fontsize=12)
plt.tight_layout()
plt.show()

### Reading the beeswarm plot

Each dot is one customer. The horizontal position shows the SHAP value (how much
that feature pushed the prediction toward attrition or toward staying). The colour
shows the feature's actual value (red = high, blue = low).

- A feature where high values (red dots) cluster on the **right** means: higher
  feature value → higher attrition risk.
- A feature where high values cluster on the **left** means: higher feature
  value → lower attrition risk.
- Wide horizontal spread means the feature has a **large effect** on predictions.

Compare this ranking with the gain-based importance from Section 10. SHAP often
reveals **direction** that gain-based importance cannot show.

In [ ]:
# ── Dependence plot: how a single feature affects predictions ────────
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Use column index positions for robust dependence plots
feat_names = list(X_test.columns)

# Pick two important features: age (usually top) and is_active_member
dep_features = ["age", "num_of_products"]
for i, feat in enumerate(dep_features):
    idx = feat_names.index(feat)
    shap.dependence_plot(idx, shap_values_unc.values, X_test.values,
                         feature_names=feat_names, ax=axes[i], show=False)
    axes[i].set_title(f"SHAP Dependence — {feat}")

plt.tight_layout()
plt.show()

### SHAP comparison: unconstrained vs monotonicity-constrained

How does monotonicity change the way the model uses features? Let us compare
the SHAP summaries side by side.

In [ ]:
# ── SHAP for the constrained model ───────────────────────────────────
explainer_con = shap.TreeExplainer(xgb_constrained)
shap_values_con = explainer_con(X_test)

# ── Side-by-side SHAP summaries ─────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(18, 6))

plt.sca(axes[0])
shap.summary_plot(shap_values_unc, X_test, show=False, plot_size=None)
axes[0].set_title("Unconstrained XGBoost", fontsize=11)

plt.sca(axes[1])
shap.summary_plot(shap_values_con, X_test, show=False, plot_size=None)
axes[1].set_title("Monotonicity-Constrained XGBoost", fontsize=11)

plt.tight_layout()
plt.show()

In [ ]:
# ── Individual explanation: waterfall for a single customer ──────────
# Pick a customer who was flagged as high risk by the constrained model
high_risk_idx = np.where(y_prob_constrained >= 0.50)[0]
if len(high_risk_idx) > 0:
    example_idx = high_risk_idx[0]
else:
    example_idx = 0

print(f"Explaining prediction for test-set customer index {example_idx}")
print(f"  Predicted probability: {y_prob_constrained[example_idx]:.3f}")
print(f"  Actual label: {'attrited' if y_test.iloc[example_idx] == 1 else 'stayed'}")
print()

shap.waterfall_plot(shap_values_con[example_idx], show=False)
plt.title(f"SHAP Waterfall — Customer #{example_idx}", fontsize=11)
plt.tight_layout()
plt.show()

### How to use SHAP in a business context

| Business question | SHAP tool | What to show |
|------------------|-----------|--------------|
| "Why was this customer flagged?" | Waterfall plot | Top 5 features pushing this prediction up/down |
| "What drives attrition overall?" | Summary (beeswarm) plot | Global feature ranking with direction and magnitude |
| "Is the model's use of age reasonable?" | Dependence plot | How age's contribution varies across the customer base |
| "Did monotonicity constraints change anything?" | Side-by-side summaries | Compare attribution patterns between constrained and unconstrained models |

### Limitations

- **SHAP is computationally expensive** — for very large datasets, approximate methods
  (`shap.Explainer` with `algorithm="auto"`) may be needed.
- **Correlated features** can mislead — if two features are highly correlated, SHAP
  may attribute importance to either one somewhat arbitrarily.
- **SHAP explains the model, not the world** — it tells you what the model learned,
  not whether the learned pattern is causal or stable.

> **Key insight:** in regulated finance, the ability to explain *why* a decision was
> made is not optional — it is a governance requirement. SHAP provides a principled,
> model-consistent framework for that explanation.

---

## Section 14 — From model result to business recommendation

### Executive recommendation

| Element | Details |
|---------|--------|
| **Recommendation** | Deploy the tuned XGBoost model as the primary attrition-risk scoring engine, using a classification threshold of **0.35** (illustrative — adjust to actual cost structure). Consider the monotonicity-constrained variant (Section 12) if regulatory interpretability is a priority. |
| **Expected benefit** | At threshold 0.35 the model identifies approximately **70 %** of at-risk customers (recall ≈ 0.71) with AUC-ROC ≈ 0.91, enabling proactive retention outreach before attrition occurs. SHAP explanations (Section 13) support individual-level justification for flagged customers. |
| **Main risk** | The model has been validated on a single train/test split of a classroom-adapted dataset with no out-of-time validation. Performance on truly future data is unverified; probability calibration has not been tested. |
| **Next steps** | 1) Validate on a recent time period (out-of-time validation). 2) Pilot with a small customer segment to measure real-world lift. 3) Calibrate predicted probabilities. 4) Establish a monitoring dashboard for data drift and model degradation. |

> **Exercise:** write your own four-sentence version of this recommendation
> before reading on. Focus on: what to do, why, what could go wrong, and what
> to check first.

---

## Section 15 — Production awareness: monitoring, drift and retraining

Deploying a model is not the end — it is the beginning of a new responsibility.

### Things that can go wrong after deployment

| Risk | Description | Example |
|------|------------|---------|
| **Data drift** | The input data distribution changes over time | A new product launch changes customer behaviour patterns |
| **Concept drift** | The relationship between features and the target changes | Attrition drivers shift from price sensitivity to service quality |
| **Model degradation** | Performance drops gradually without a single breaking event | The model still runs, but recall falls from 0.75 to 0.50 over 6 months |
| **Data quality issues** | Missing values, wrong formats, or stale data | A data pipeline breaks and `digital_usage_proxy` becomes all zeros |

### What responsible model management looks like

1. **Monitor key metrics** — track precision, recall and AUC on live predictions.
   Set up alerts when performance drops below a threshold (e.g., AUC < 0.80).
2. **Retrain periodically** — refresh the model on recent data (e.g., quarterly).
3. **Document assumptions** — record which features are used, what thresholds
   were chosen, and what business context justified the decisions.
4. **Keep a challenger model** — maintain a simple baseline (e.g., logistic
   regression) alongside the production model. If the complex model degrades
   faster, the simple one can be a fallback.
5. **Governance** — ensure compliance with internal policies and external
   regulations (e.g., GDPR, fair lending laws). Use SHAP explanations (Section 13)
   and monotonicity constraints (Section 12) to support auditability.

> *"A model is part of a decision system, not a magic answer."*

---

### Limitations of this analysis

| Limitation | Impact |
|-----------|--------|
| Single dataset (no temporal split) | We cannot verify how the model performs on truly future data |
| No feature interaction analysis | Some feature combinations may matter more than individual features |
| No cost-sensitive learning | We chose the threshold post-hoc; a cost-sensitive objective could be more principled |
| Classroom-adapted labels | The target variable uses derived teaching features; no temporal deployment validation has been performed |
| No A/B testing | The true lift of the model can only be measured through a controlled experiment |

These limitations are normal at the **prototype stage**. The goal is awareness,
not perfection.

---

### Self-practice tasks

1. **Threshold comparison:** pick a third threshold (e.g., 0.25 or 0.45) and
   compute precision, recall and F1. Explain in 2–3 sentences when this
   threshold would make business sense.
2. **Accuracy critique:** write a short paragraph explaining why accuracy alone
   is insufficient for this business problem.
3. **Simpler model test:** re-run the XGBoost grid search with `max_depth` ∈
   {2, 3} only. Does the simpler model lose much AUC? Would you still
   recommend it? Justify.
4. **Stakeholder memo:** write a four-sentence "go / no-go / pilot"
   recommendation for a bank manager who has never seen a confusion matrix.

---

**End of Notebook 06.**

This closes the core modelling cycle of Topic 2. In the next topic we
explore more advanced techniques — but the validation discipline, threshold
awareness and business framing introduced here apply to **every model you
will ever build**.

---

**Project chain:** NB04 (build table) → NB04b (treat data) → NB05 (baseline) → NB05b (trees) → **NB06 (validate & interpret)**  
**Current position:** NB06 — the capstone of the core modelling workflow